**Desadio que tem como objetivo classificar se o aluno est? participando do enem 2016 como treineiro**

## Adicionando bibliotecas e dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import plot_confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

%matplotlib inline

**Ap?s an?lise ao dicion?rio foram escolhidas as respectivas features para classifica??o do aluno treineiro.**

- **NU_IDADE:** Idade do participante

- **TP_ST_CONCLUSAO:** Situa??o de conclus?o do Ensino M?dio
    - 1: J? conclu? o Ensino M?dio
    - 2: Estou cursando e concluirei o Ensino M?dio em 2016
    - 3: Estou cursando e concluirei o Ensino M?dio ap?s 2016
    - 4: N?o conclu? e n?o estou cursando o Ensino M?dio
    
- **TP_ANO_CONCLUIU:** Ano de Conclus?o do Ensino M?dio
    - 0: N?o informado
    - 1: 2015
    - 2: 2014
    - 3: 2013
    - 4: 2012
    - 5: 2011
    - 6: 2010
    - 7: 2009
    - 8: 2008
    - 9: 2007
    - 10: Anterior a 2007

- **IN_TREINEIRO:** Indica se o inscrito fez a prova com intuito de apenas treinar seus conhecimentos
    - 0: N?o ? treineiro
    - 1: ? treineiro

In [ ]:
cols_train = ['NU_IDADE', 'TP_ST_CONCLUSAO', 'TP_ANO_CONCLUIU', 'IN_TREINEIRO']
cols_test = ['NU_INSCRICAO', 'NU_IDADE', 'TP_ST_CONCLUSAO', 'TP_ANO_CONCLUIU']

In [ ]:
df_train = pd.read_csv('train.csv', usecols= cols_train)
df_test = pd.read_csv('test.csv', usecols= cols_test)

## An?lise Explorat?ria

In [ ]:
df_train.head()

In [ ]:
df_train.info()

In [ ]:
plt.figure(figsize=(12,10))

plt.subplot(3, 1, 1)
sns.distplot(df_train['NU_IDADE']);

plt.subplot(3, 1, 2)
sns.countplot(x = 'TP_ST_CONCLUSAO', hue= 'IN_TREINEIRO', data= df_train, );

plt.subplot(3, 1, 3)
sns.countplot(x= 'TP_ANO_CONCLUIU', data= df_train);

## Verificar balanceamento dos dados

In [ ]:
df_train['IN_TREINEIRO'].value_counts()

Observado que existe um desbalanceamento, existem poucos alunos treineiros

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(df_train['IN_TREINEIRO']);

Balanceamento dos dados utilizando o met?do SMOTE com a estrat?gia de aumentar a quantidade de dados da classe minorit?ria

In [ ]:
smote = SMOTE(sampling_strategy="minority")

In [ ]:
X_smote, y_smote = smote.fit_resample(X= df_train.iloc[:, :-1], y= df_train.iloc[:, -1])

Verifica??o e ilustra??o dos dados ap?s balanceamento

In [ ]:
y_smote.value_counts()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(y_smote);

## Machine Learning

Separa??o dos dados

In [ ]:
data = X_smote
target = y_smote
test = df_test.drop(labels= 'NU_INSCRICAO', axis= 1)

### Random Forest Classifier

In [ ]:
rfc = RandomForestClassifier(max_depth= 100, n_jobs= 3)

In [ ]:
#treinando os dados
rfc.fit(data, target)

### Verifica??o das m?tricas

In [ ]:
data_pred = rfc.predict(data)

In [ ]:
print(classification_report(target, data_pred))

In [ ]:
print(pd.crosstab(target, data_pred, rownames=['real'], colnames=['      predito']))

In [ ]:
plt.figaspect
plot_confusion_matrix(rfc,data, target);

### Predizendo os dados de teste

In [ ]:
result = rfc.predict(test)

## Exportando resultado

In [ ]:
answer = pd.DataFrame(data= df_test['NU_INSCRICAO'])
answer['IN_TREINEIRO'] = result

In [ ]:
answer.to_csv('answer.csv', index= False)

In [ ]:
verificad = pd.read_csv('answer.csv')

In [ ]:
verificad

In [ ]:
verificad['IN_TREINEIRO'].value_counts()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x= 'IN_TREINEIRO', data= verificad)
plt.title('Classifica??o da situa??o do aluno como treineiro', )
plt.legend(labels=['N?o treineiro'])

Com esta abordagem obtive um score de 98.73%